### Load Data

In [1]:
import numpy as np

from helpers import load_csv_data

x_train, x_test, y_train, train_ids, test_ids = load_csv_data(
    "dataset",
    sub_sample=False
)

In [2]:
# Check the shape of the loaded data
print("Training data shape:", x_train.shape)
print("Test data shape:", x_test.shape)
print("Training labels shape:", y_train.shape)

print(train_ids[:5])  # Print the first 5 training IDs
print(test_ids[:5])   # Print the first 5 test IDs
print(test_ids[-5:])  # Print the last 5 test IDs

Training data shape: (328135, 321)
Test data shape: (109379, 321)
Training labels shape: (328135,)
[0 1 2 3 4]
[328135 328136 328137 328138 328139]
[437509 437510 437511 437512 437513]


In [3]:
# Check class imbalance in the training labels
unique, counts = np.unique(y_train, return_counts=True)

print("Label distribution:")
print("-" * 40)

for label, count in zip(unique, counts):
    print(
        f"Label {label}: {count} samples "
        f"({100 * count / len(y_train):.2f}%)"
    )

baseline_accuracy = np.max(counts) / len(y_train)

print(
    f"\nMajority-class baseline accuracy: "
    f"{baseline_accuracy:.4f}"
)

Label distribution:
----------------------------------------
Label -1: 299160 samples (91.17%)
Label 1: 28975 samples (8.83%)

Majority-class baseline accuracy: 0.9117


### Split training data

In [4]:
# We will use the same code we used for ex_3

def split_data(x, y, ratio, seed=1):
    """
    split the dataset based on the split ratio. If ratio is 0.8
    you will have 80% of your data set dedicated to training
    and the rest dedicated to testing. If ratio times the number of samples is not round
    you can use np.floor. Also check the documentation for np.random.permutation,
    it could be useful.

    Args:
        x: numpy array of shape (N,), N is the number of samples.
        y: numpy array of shape (N,).
        ratio: scalar in [0,1]
        seed: integer.

    Returns:
        x_tr: numpy array containing the train data.
        x_te: numpy array containing the test data.
        y_tr: numpy array containing the train labels.
        y_te: numpy array containing the test labels.

    >>> split_data(np.arange(13), np.arange(13), 0.8, 1)
    (array([ 2,  3,  4, 10,  1,  6,  0,  7, 12,  9]), array([ 8, 11,  5]), array([ 2,  3,  4, 10,  1,  6,  0,  7, 12,  9]), array([ 8, 11,  5]))
    """
    # set seed
    np.random.seed(seed)
    # ***************************************************
    # INSERT YOUR CODE HERE
    # split the data based on the given ratio: TODO
    # ***************************************************
    # Randomly shuffle the indices
    indices = np.random.permutation(len(x))

    # Number of training samples
    split_sample = int(np.floor(ratio * len(x)))

    # Split the shuffled indices
    train_indices = indices[:split_sample]
    test_indices = indices[split_sample:]

    # Use SAME indices for x and y
    x_tr = x[train_indices]
    x_te = x[test_indices]
    y_tr = y[train_indices]
    y_te = y[test_indices]

    return x_tr, x_te, y_tr, y_te

In [5]:
train_X, test_X, train_y, test_y = split_data(x_train, y_train, 0.8, seed=2026)

print("Training data shape after split:", train_X.shape)
print("Test data shape after split:", test_X.shape)

Training data shape after split: (262508, 321)
Test data shape after split: (65627, 321)


In [6]:
# Check the train/test split imbalance
def print_label_distribution(y, name):
    values, counts = np.unique(
        y,
        return_counts=True
    )

    print(f"\n{name}")
    print("-" * 40)

    for value, count in zip(values, counts):
        print(
            f"Label {value}: {count} "
            f"({100 * count / len(y):.2f}%)"
        )


print_label_distribution(
    train_y,
    "TRAINING SET"
)

print_label_distribution(
    test_y,
    "VALIDATION SET"
)


TRAINING SET
----------------------------------------
Label -1: 239487 (91.23%)
Label 1: 23021 (8.77%)

VALIDATION SET
----------------------------------------
Label -1: 59673 (90.93%)
Label 1: 5954 (9.07%)


In [7]:
feature_summary = []

num_samples, num_features = train_X.shape

for j in range(num_features):

    col = train_X[:, j]

    missing_mask = np.isnan(col)
    observed = col[~missing_mask]

    n_missing = np.sum(missing_mask)
    n_observed = len(observed)

    missing_fraction = n_missing / num_samples

    if n_observed == 0:
        n_unique = 0
        dominant_value = np.nan
        dominant_fraction = np.nan
        min_value = np.nan
        max_value = np.nan

    else:
        values, counts = np.unique(
            observed,
            return_counts=True
        )

        n_unique = len(values)

        dominant_index = np.argmax(counts)

        dominant_value = values[dominant_index]

        dominant_fraction = (
            counts[dominant_index]
            / n_observed
        )

        min_value = np.min(observed)
        max_value = np.max(observed)

    feature_summary.append({
        "index": j,
        "n_missing": n_missing,
        "n_observed": n_observed,
        "missing_fraction": missing_fraction,
        "n_unique": n_unique,
        "dominant_value": dominant_value,
        "dominant_fraction": dominant_fraction,
        "min": min_value,
        "max": max_value
    })


print(
    f"{'Feature':>8} "
    f"{'Missing%':>10} "
    f"{'Unique':>8} "
    f"{'Dominant':>12} "
    f"{'Dom.%':>10} "
    f"{'Min':>10} "
    f"{'Max':>10}"
)

print("-" * 85)

for s in feature_summary:

    dominant_percentage = (
        100 * s["dominant_fraction"]
        if not np.isnan(s["dominant_fraction"])
        else np.nan
    )

    print(
        f"{s['index']:8d} "
        f"{100*s['missing_fraction']:10.2f} "
        f"{s['n_unique']:8d} "
        f"{s['dominant_value']:12.2f} "
        f"{dominant_percentage:10.2f} "
        f"{s['min']:10.2f} "
        f"{s['max']:10.2f}"
    )

 Feature   Missing%   Unique     Dominant      Dom.%        Min        Max
-------------------------------------------------------------------------------------
       0       0.00       53        20.00       5.30       1.00      72.00
       1       0.00       12         1.00      10.02       1.00      12.00
       2       0.00      414   6022015.00       0.49 1012016.00 12312015.00
       3       0.00       12         3.00       9.58       1.00      12.00
       4       0.00       31         9.00       4.22       1.00      31.00
       5       0.00        2      2015.00      97.52    2015.00    2016.00
       6       0.00        2      1100.00      84.95    1100.00    1200.00
       7       0.00    20750 2015000426.00       0.02 2015000001.00 2015023235.00
       8       0.00    20750 2015000426.00       0.02 2015000001.00 2015023235.00
       9      42.46        1         1.00     100.00       1.00       1.00
      10      42.46        2         1.00      99.98       1.00       2.00

In [8]:
all_missing_features = [
    s["index"]
    for s in feature_summary
    if s["n_observed"] == 0
]

constant_features = [
    s["index"]
    for s in feature_summary
    if s["n_unique"] == 1
]

missing_90 = [
    s["index"]
    for s in feature_summary
    if s["missing_fraction"] >= 0.90
]

missing_95 = [
    s["index"]
    for s in feature_summary
    if s["missing_fraction"] >= 0.95
]

missing_99 = [
    s["index"]
    for s in feature_summary
    if s["missing_fraction"] >= 0.99
]

dominant_99 = [
    s["index"]
    for s in feature_summary
    if (
        not np.isnan(s["dominant_fraction"])
        and s["dominant_fraction"] >= 0.99
    )
]


print("All-missing features:")
print(all_missing_features)
print("Count:", len(all_missing_features))

print("\nConstant when observed:")
print(constant_features)
print("Count:", len(constant_features))

print("\n>= 90% missing:")
print(missing_90)
print("Count:", len(missing_90))

print("\n>= 95% missing:")
print(missing_95)
print("Count:", len(missing_95))

print("\n>= 99% missing:")
print(missing_99)
print("Count:", len(missing_99))

print("\n>= 99% same observed value:")
print(dominant_99)
print("Count:", len(dominant_99))

All-missing features:
[]
Count: 0

Constant when observed:
[9, 11, 12, 18, 19, 22]
Count: 6

>= 90% missing:
[11, 14, 22, 55, 109, 110, 111, 112, 113, 114, 115, 116, 117, 119, 120, 121, 122, 123, 124, 125, 127, 128, 129, 130, 131, 132, 133, 134, 135, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 180, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 194, 195, 196, 197, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215]
Count: 99

>= 95% missing:
[11, 14, 22, 55, 127, 128, 129, 130, 131, 132, 133, 134, 135, 137, 138, 139, 140, 141, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 167, 168, 171, 173, 175, 177, 178, 180, 184, 185, 186, 187, 188, 189, 190, 191, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215]
Count: 68

>= 99% missing:
[11, 14,

In [9]:
print(
    f"{'Feat':>5} "
    f"{'Miss%':>9} "
    f"{'N miss':>9} "
    f"{'N obs':>9} "
    f"{'P(+|miss)':>12} "
    f"{'P(+|obs)':>12} "
    f"{'Diff':>10}"
)

print("-" * 85)

for j in constant_features:

    col = train_X[:, j]

    missing_mask = np.isnan(col)
    observed_mask = ~missing_mask

    n_missing = np.sum(missing_mask)
    n_observed = np.sum(observed_mask)

    if n_missing > 0:
        p_missing = np.mean(
            train_y[missing_mask] == 1
        )
    else:
        p_missing = np.nan

    if n_observed > 0:
        p_observed = np.mean(
            train_y[observed_mask] == 1
        )
    else:
        p_observed = np.nan

    print(
        f"{j:5d} "
        f"{100*n_missing/num_samples:9.2f} "
        f"{n_missing:9d} "
        f"{n_observed:9d} "
        f"{p_missing:12.4f} "
        f"{p_observed:12.4f} "
        f"{p_missing-p_observed:10.4f}"
    )

 Feat     Miss%    N miss     N obs    P(+|miss)     P(+|obs)       Diff
-------------------------------------------------------------------------------------
    9     42.46    111469    151039       0.0619       0.1067    -0.0448
   11     99.99    262485        23       0.0877       0.0435     0.0442
   12     42.46    111470    151038       0.0619       0.1067    -0.0448
   18     57.54    151039    111469       0.1067       0.0619     0.0448
   19     57.54    151039    111469       0.1067       0.0619     0.0448
   22     99.76    261875       633       0.0879       0.0079     0.0800


In [10]:
missing_pattern_groups = {}

for j in range(num_features):

    missing_mask = np.isnan(
        train_X[:, j]
    )

    # Convert boolean mask into compact bytes
    packed_mask = np.packbits(
        missing_mask
    )

    key = packed_mask.tobytes()

    if key not in missing_pattern_groups:
        missing_pattern_groups[key] = []

    missing_pattern_groups[key].append(j)


shared_missing_groups = [
    group
    for group
    in missing_pattern_groups.values()
    if len(group) > 1
]

shared_missing_groups.sort(
    key=len,
    reverse=True
)


print(
    "Number of shared missingness groups:",
    len(shared_missing_groups)
)

print("\nLargest groups:")
print("=" * 60)

for i, group in enumerate(
    shared_missing_groups[:30],
    start=1
):

    print(
        f"Group {i}: "
        f"{len(group)} features"
    )

    print(group)

Number of shared missingness groups: 18

Largest groups:
Group 1: 82 features
[0, 1, 2, 3, 4, 5, 6, 7, 8, 28, 30, 31, 36, 39, 40, 43, 44, 46, 47, 50, 51, 52, 53, 58, 216, 219, 221, 222, 227, 229, 230, 231, 232, 233, 235, 236, 237, 239, 240, 241, 242, 243, 244, 246, 247, 248, 249, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 272, 273, 274, 275, 278, 279, 280, 281, 282, 283, 284, 287, 288, 298, 305, 306, 307, 308, 309, 310, 311, 312, 316, 317]
Group 2: 6 features
[111, 112, 113, 115, 116, 117]
Group 3: 5 features
[9, 10, 13, 54, 56]
Group 4: 4 features
[18, 19, 21, 23]
Group 5: 4 features
[238, 313, 314, 315]
Group 6: 3 features
[37, 38, 234]
Group 7: 3 features
[149, 150, 151]
Group 8: 2 features
[11, 14]
Group 9: 2 features
[32, 33]
Group 10: 2 features
[104, 320]
Group 11: 2 features
[128, 129]
Group 12: 2 features
[133, 134]
Group 13: 2 features
[147, 148]
Group 14: 2 features
[153, 154]
Group 15: 2 features
[186, 187]
Group 16: 2 features
[224, 225]
Group 17: 2 features
[2

In [11]:
MAX_UNIQUE = 10
MIN_CATEGORY_SIZE = 100

low_cardinality_features = [
    s["index"]
    for s in feature_summary
    if (
        s["n_unique"] >= 2
        and s["n_unique"] <= MAX_UNIQUE
    )
]


print(
    "Number of low-cardinality features:",
    len(low_cardinality_features)
)


for j in low_cardinality_features:

    col = train_X[:, j]

    observed = col[~np.isnan(col)]

    values = np.unique(observed)

    print("\n" + "=" * 65)
    print(f"FEATURE {j}")
    print("=" * 65)

    for value in values:

        mask = col == value

        n = np.sum(mask)

        if n < MIN_CATEGORY_SIZE:
            continue

        positive_rate = np.mean(
            train_y[mask] == 1
        )

        print(
            f"value = {value:8.2f} | "
            f"N = {n:7d} | "
            f"P(+1) = {positive_rate:.4f}"
        )

    missing_mask = np.isnan(col)

    n_missing = np.sum(missing_mask)

    if n_missing >= MIN_CATEGORY_SIZE:

        positive_rate_missing = np.mean(
            train_y[missing_mask] == 1
        )

        print(
            f"{'MISSING':>16} | "
            f"N = {n_missing:7d} | "
            f"P(+1) = "
            f"{positive_rate_missing:.4f}"
        )

Number of low-cardinality features: 214

FEATURE 5
value =  2015.00 | N =  256002 | P(+1) = 0.0884
value =  2016.00 | N =    6506 | P(+1) = 0.0613

FEATURE 6
value =  1100.00 | N =  222996 | P(+1) = 0.0898
value =  1200.00 | N =   39512 | P(+1) = 0.0759

FEATURE 10
value =     1.00 | N =  151016 | P(+1) = 0.1067
         MISSING | N =  111469 | P(+1) = 0.0619

FEATURE 13
value =     1.00 | N =   68824 | P(+1) = 0.1024
value =     2.00 | N =   82215 | P(+1) = 0.1103
         MISSING | N =  111469 | P(+1) = 0.0619

FEATURE 14
         MISSING | N =  262485 | P(+1) = 0.0877

FEATURE 20
value =     1.00 | N =   55093 | P(+1) = 0.0774
value =     2.00 | N =   56375 | P(+1) = 0.0468
         MISSING | N =  151040 | P(+1) = 0.1067

FEATURE 21
value =     1.00 | N =  110836 | P(+1) = 0.0622
value =     2.00 | N =     633 | P(+1) = 0.0079
         MISSING | N =  151039 | P(+1) = 0.1067

FEATURE 23
value =     1.00 | N =  104031 | P(+1) = 0.0634
value =     2.00 | N =    7438 | P(+1) = 0.0421
  

value =     4.00 | N =   16141 | P(+1) = 0.0444
value =     7.00 | N =    3198 | P(+1) = 0.1001
value =     8.00 | N =    2272 | P(+1) = 0.0599
value =     9.00 | N =     385 | P(+1) = 0.1039

FEATURE 34
value =     1.00 | N =  105255 | P(+1) = 0.1624


value =     2.00 | N =    1975 | P(+1) = 0.0324
value =     3.00 | N =  151915 | P(+1) = 0.0370
value =     4.00 | N =    2634 | P(+1) = 0.0600
value =     7.00 | N =     492 | P(+1) = 0.1484
value =     9.00 | N =     236 | P(+1) = 0.0466

FEATURE 35
value =     1.00 | N =   87977 | P(+1) = 0.1794
value =     2.00 | N =   17110 | P(+1) = 0.0741
value =     7.00 | N =     138 | P(+1) = 0.2899
         MISSING | N =  157253 | P(+1) = 0.0377

FEATURE 36
value =     1.00 | N =  227476 | P(+1) = 0.0973
value =     2.00 | N =   29512 | P(+1) = 0.0214
value =     7.00 | N =    5429 | P(+1) = 0.0442

FEATURE 37
value =     1.00 | N =  174373 | P(+1) = 0.1144
value =     2.00 | N =   26029 | P(+1) = 0.0428
value =     3.00 | N =   15479 | P(+1) = 0.0311
value =     4.00 | N =    8104 | P(+1) = 0.0315
value =     7.00 | N =    3326 | P(+1) = 0.0992
value =     9.00 | N =     165 | P(+1) = 0.0788
         MISSING | N =   35032 | P(+1) = 0.0251

FEATURE 38
value =     1.00 | N =   94906 | P(+1) =


FEATURE 48
value =     1.00 | N =   33642 | P(+1) = 0.2199
value =     2.00 | N =    2104 | P(+1) = 0.0404
value =     3.00 | N =  221826 | P(+1) = 0.0669
value =     4.00 | N =    4565 | P(+1) = 0.1446


value =     7.00 | N =     310 | P(+1) = 0.1161

FEATURE 50
value =     1.00 | N =  110902 | P(+1) = 0.1111
value =     2.00 | N =  151606 | P(+1) = 0.0706

FEATURE 51
value =     1.00 | N =  139035 | P(+1) = 0.0791
value =     2.00 | N =   35449 | P(+1) = 0.1082
value =     3.00 | N =   33184 | P(+1) = 0.1718
value =     4.00 | N =    5257 | P(+1) = 0.1025
value =     5.00 | N =   40359 | P(+1) = 0.0383
value =     6.00 | N =    7461 | P(+1) = 0.0417
value =     9.00 | N =    1763 | P(+1) = 0.0522

FEATURE 52
value =     1.00 | N =     360 | P(+1) = 0.1417
value =     2.00 | N =    6485 | P(+1) = 0.1611
value =     3.00 | N =   13323 | P(+1) = 0.1414
value =     4.00 | N =   73161 | P(+1) = 0.1040
value =     5.00 | N =   71614 | P(+1) = 0.0878
value =     6.00 | N =   96568 | P(+1) = 0.0627
value =     9.00 | N =     997 | P(+1) = 0.0853

FEATURE 53
value =     1.00 | N =  188474 | P(+1) = 0.0894
value =     2.00 | N =   59883 | P(+1) = 0.0849
value =     3.00 | N =   12342 | P(+1) =

...

value =     1.00 | N =   14532 | P(+1) = 0.0053
value =     2.00 | N =   25446 | P(+1) = 0.0101
value =     3.00 | N =   30330 | P(+1) = 0.0190
value =     4.00 | N =   42892 | P(+1) = 0.0471
value =     5.00 | N =   58765 | P(+1) = 0.0850
value =     6.00 | N =   90543 | P(+1) = 0.1667

FEATURE 254
value =     1.00 | N =    3975 | P(+1) = 0.0858
value =     2.00 | N =   78250 | P(+1) = 0.0667
value =     3.00 | N =   87444 | P(+1) = 0.0928
value =     4.00 | N =   71134 | P(+1) = 0.1128
         MISSING | N =   21705 | P(+1) = 0.0612

FEATURE 255
value =     1.00 | N =   82225 | P(+1) = 0.0676
value =     2.00 | N =  158578 | P(+1) = 0.1017
value =     9.00 | N =   21705 | P(+1) = 0.0612

FEATURE 256
value =     1.00 | N =  192360 | P(+1) = 0.1075
value =     2.00 | N =   27568 | P(+1) = 0.0424
value =     3.00 | N =   24380 | P(+1) = 0.0256
value =     4.00 | N =   10288 | P(+1) = 0.0270
value =     5.00 | N =    3815 | P(+1) = 0.0286
value =     6.00 | N =    1901 | P(+1) = 0.0274
v

value =     2.00 | N =    2064 | P(+1) = 0.1260
value =     3.00 | N =    1272 | P(+1) = 0.0904
value =     4.00 | N =   19690 | P(+1) = 0.0809

FEATURE 274
value =     0.00 | N =   25785 | P(+1) = 0.0890


value =     1.00 | N =  236723 | P(+1) = 0.0876

FEATURE 275
value =     0.00 | N =   30195 | P(+1) = 0.0946
value =     1.00 | N =  232313 | P(+1) = 0.0868

FEATURE 278
value =     1.00 | N =  147493 | P(+1) = 0.0846
value =     2.00 | N =   89230 | P(+1) = 0.0925
value =     9.00 | N =   25785 | P(+1) = 0.0890

FEATURE 279
value =     1.00 | N =  184512 | P(+1) = 0.0822
value =     2.00 | N =   47801 | P(+1) = 0.1044
value =     9.00 | N =   30195 | P(+1) = 0.0946

FEATURE 280
value =     1.00 | N =  262448 | P(+1) = 0.0877

FEATURE 281
value =     1.00 | N =  262478 | P(+1) = 0.0877

FEATURE 282
value =     0.00 | N =  236663 | P(+1) = 0.0875
value =     1.00 | N =   25785 | P(+1) = 0.0890

FEATURE 283
value =     0.00 | N =  232283 | P(+1) = 0.0868
value =     1.00 | N =   30195 | P(+1) = 0.0946

FEATURE 284
value =     1.00 | N =  176192 | P(+1) = 0.0744
value =     2.00 | N =   63746 | P(+1) = 0.1270
value =     9.00 | N =   22570 | P(+1) = 0.0803

FEATURE 289
value =     0.00 | 

value =     1.00 | N =   78054 | P(+1) = 0.0814
value =     2.00 | N =   84835 | P(+1) = 0.0653
value =     3.00 | N =   66446 | P(+1) = 0.1255
value =     9.00 | N =   33173 | P(+1) = 0.0842



FEATURE 309
value =     1.00 | N =   78054 | P(+1) = 0.0814
value =     2.00 | N =  151281 | P(+1) = 0.0917
value =     9.00 | N =   33173 | P(+1) = 0.0842

FEATURE 310
value =     1.00 | N =   65285 | P(+1) = 0.0684
value =     2.00 | N =  171126 | P(+1) = 0.0961
value =     9.00 | N =   26097 | P(+1) = 0.0810

FEATURE 311
value =     1.00 | N =   44559 | P(+1) = 0.0632
value =     2.00 | N =   75999 | P(+1) = 0.0829
value =     3.00 | N =   18562 | P(+1) = 0.0800
value =     4.00 | N =   89233 | P(+1) = 0.1066
value =     9.00 | N =   34155 | P(+1) = 0.0851

FEATURE 312
value =     1.00 | N =   44559 | P(+1) = 0.0632
value =     2.00 | N =  183794 | P(+1) = 0.0941
value =     9.00 | N =   34155 | P(+1) = 0.0851

FEATURE 313
value =     1.00 | N =   39571 | P(+1) = 0.1875
value =     2.00 | N =   40419 | P(+1) = 0.1211
value =     3.00 | N =  173401 | P(+1) = 0.0540
value =     9.00 | N =    7624 | P(+1) = 0.1542
         MISSING | N =    1493 | P(+1) = 0.1139

FEATURE 314
value =   

In [12]:
candidate_special_codes = np.array([
    7, 8, 9,
    77, 88, 99,
    777, 888, 999,
    7777, 8888, 9999,
    77777, 88888, 99999
], dtype=float)


special_code_features = []

for j in range(num_features):

    col = train_X[:, j]

    observed = col[
        ~np.isnan(col)
    ]

    codes_found = []

    for code in candidate_special_codes:

        if np.any(observed == code):
            codes_found.append(code)

    if len(codes_found) > 0:
        special_code_features.append(
            (j, codes_found)
        )


print(
    "Features containing candidate "
    "special codes:"
)

print("-" * 60)

for feature, codes in special_code_features:

    print(
        f"Feature {feature:3d}: "
        f"{codes}"
    )

Features containing candidate special codes:
------------------------------------------------------------
Feature   0: [np.float64(8.0), np.float64(9.0)]
Feature   1: [np.float64(7.0), np.float64(8.0), np.float64(9.0)]
Feature   3: [np.float64(7.0), np.float64(8.0), np.float64(9.0)]
Feature   4: [np.float64(7.0), np.float64(8.0), np.float64(9.0)]
Feature  15: [np.float64(7.0), np.float64(8.0), np.float64(9.0)]
Feature  16: [np.float64(7.0), np.float64(8.0), np.float64(9.0)]
Feature  17: [np.float64(7.0), np.float64(8.0), np.float64(9.0)]
Feature  24: [np.float64(7.0), np.float64(9.0)]
Feature  25: [np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(77.0), np.float64(99.0)]
Feature  26: [np.float64(7.0), np.float64(9.0)]
Feature  27: [np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(77.0), np.float64(88.0), np.float64(99.0)]
Feature  28: [np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(77.0), np.float64(88.0), np.float64(99.0)]
Feature  29: [np.floa

In [13]:
print("=" * 60)
print("EDA SUMMARY")
print("=" * 60)

print(
    "Total features:",
    num_features
)

print(
    "All-missing features:",
    len(all_missing_features)
)

print(
    "Constant when observed:",
    len(constant_features)
)

print(
    ">= 90% missing:",
    len(missing_90)
)

print(
    ">= 95% missing:",
    len(missing_95)
)

print(
    ">= 99% missing:",
    len(missing_99)
)

print(
    ">= 99% dominated:",
    len(dominant_99)
)

print(
    "Low-cardinality features:",
    len(low_cardinality_features)
)

print(
    "Shared missingness groups:",
    len(shared_missing_groups)
)

print(
    "Features containing candidate "
    "special codes:",
    len(special_code_features)
)

EDA SUMMARY
Total features: 321
All-missing features: 0
Constant when observed: 6
>= 90% missing: 99
>= 95% missing: 68
>= 99% missing: 30
>= 99% dominated: 10
Low-cardinality features: 214
Shared missingness groups: 18
Features containing candidate special codes: 273


In [14]:
# ============================================================
# COMPLETE BRFSS PREPROCESSING PIPELINE
# ============================================================
#
# Uses the variables already present in your notebook:
#
#   x_train  -> complete labelled feature matrix
#   train_X  -> 80% training split
#   test_X   -> 20% local validation split
#   x_test   -> competition test set
#
# After this cell you will have:
#
#   train_X_processed
#   test_X_processed
#   x_test_processed
#   processed_feature_names
#
# IMPORTANT:
#   Everything that must be learned from data (medians, means,
#   standard deviations, categories, etc.) is learned ONLY from
#   train_X. This avoids validation/test leakage.
#
# Encoding philosophy:
#
#   1. Survey metadata / IDs / sampling weights:
#        -> DROP
#
#   2. Constant-when-observed features:
#        -> keep only a missingness indicator
#
#   3. Binary/categorical survey variables:
#        -> dummy/one-hot encoding
#        -> missing/unknown is its own category
#
#      WHY ONE-HOT YES/NO/MISSING?
#      --------------------------------
#      Yes/No/Missing are categorical states. We therefore do
#      NOT need to pretend that missing has some numerical value.
#
#      We use k-1 dummy encoding. For example:
#
#          raw          NO    MISSING
#          Yes           0       0       <- reference category
#          No            1       0
#          Missing       0       1
#
#      This contains the same information as full 3-column
#      one-hot encoding without adding a redundant column.
#
#   4. Ordinal variables:
#        -> preserve their meaningful order
#        -> invalid/unknown response -> NaN
#        -> training median imputation
#        -> missingness indicator
#        -> standardization
#
#   5. Numerical variables:
#        -> decode feature-specific special values where known
#        -> training median imputation
#        -> missingness indicator
#        -> standardization
#
#   6. >=99% missing features:
#        -> DROP in baseline v1
#
#      We can later perform an ablation where we retain them.
#
# ============================================================

import csv
import numpy as np


# ============================================================
# 0. GET THE EXACT FEATURE NAMES
# ============================================================
#
# load_csv_data() removes the Id column, so we read only the
# header here and remove "Id".
#
# If your x_train.csv lives somewhere else, ONLY change this
# path.
# ============================================================

X_TRAIN_CSV_PATH = "dataset/x_train.csv"

with open(X_TRAIN_CSV_PATH, "r") as f:
    reader = csv.reader(f)
    csv_header = next(reader)

# First CSV column is Id and is not present in x_train/train_X.
feature_names = csv_header[1:]

assert len(feature_names) == x_train.shape[1], (
    f"Header has {len(feature_names)} features but "
    f"x_train has {x_train.shape[1]} columns."
)

name_to_index = {
    name: i
    for i, name in enumerate(feature_names)
}

print("Raw number of features:", len(feature_names))


# ============================================================
# 1. FEATURES TO DROP
# ============================================================
#
# These are survey identifiers, interview metadata,
# sampling/weighting variables, or redundant representations.
#
# We do NOT want the model learning things such as survey
# sequence number or sampling weight as if they were patient
# health characteristics.
# ============================================================

DROP_FEATURES = {
    # --------------------------------------------------------
    # Survey / interview metadata
    # --------------------------------------------------------
    "_STATE",
    "FMONTH",
    "IDATE",
    "IMONTH",
    "IDAY",
    "IYEAR",
    "DISPCODE",
    "SEQNO",
    "_PSU",
    "QSTVER",
    "QSTLANG",

    # --------------------------------------------------------
    # Sampling / weighting variables
    # --------------------------------------------------------
    "_STSTR",
    "_STRWT",
    "_RAWRAKE",
    "_WT2RAKE",
    "_CLLCPWT",
    "_DUALCOR",
    "_LLCPWT",

    # --------------------------------------------------------
    # Almost-empty administrative variables
    # --------------------------------------------------------
    "COLGHOUS",
    "LADULT",
    "CCLGHOUS",

    # --------------------------------------------------------
    # Redundant AGE representations.
    #
    # We keep _AGE80 as the main numerical age representation.
    # --------------------------------------------------------
    "_AGEG5YR",
    "_AGE65YR",
    "_AGE_G",

    # --------------------------------------------------------
    # Redundant BMI representations.
    #
    # We keep _BMI5 as the main BMI representation.
    # --------------------------------------------------------
    "_BMI5CAT",
    "_RFBMI5",

    # --------------------------------------------------------
    # Alternative height/weight representations.
    #
    # We retain the cleaner derived BMI representation for
    # baseline v1 rather than feeding several strongly related
    # versions of the same information.
    # --------------------------------------------------------
    "HTIN4", # TODO These specifically we may want tokeep as well
    "HTM4",
    "WTKG3",

    # --------------------------------------------------------
    # Raw answers that pack a UNIT and an AMOUNT into one
    # number, e.g. ALCDAY5:
    #
    #       107 = 7 days per WEEK
    #       202 = 2 days per MONTH
    #       888 = none
    #
    # Read as a plain number the order is meaningless (even
    # reversed). The dataset already contains decoded versions
    # in a single unit, which we keep:
    #
    #       ALCDAY5              -> DROCDY3_, _DRNKWEK
    #       fruit / vegetables   -> FTJUDA1_, FRUTDA1_, BEANDAY_,
    #                               GRENDAY_, ORNGDAY_, VEGEDA1_
    #       exercise             -> PAFREQ1_, PAFREQ2_, PADUR1_,
    #                               PADUR2_, STRFREQ_
    #       WEIGHT2, HEIGHT3     -> _BMI5
    #                               (9000+ = metric units)
    # --------------------------------------------------------
    "ALCDAY5",
    "FRUITJU1",
    "FRUIT1",
    "FVBEANS",
    "FVGREEN",
    "FVORANG",
    "VEGETAB1",
    "EXEROFT1",
    "EXEROFT2",
    "EXERHMM1",
    "EXERHMM2",
    "STRENGTH",
    "WEIGHT2",
    "HEIGHT3",
}


# Only retain names that actually exist in this dataset.
DROP_FEATURES &= set(feature_names)


# ============================================================
# 2. CONSTANT-WHEN-OBSERVED FEATURES
# ============================================================
#
# From our EDA:
#
#   CTELENUM
#   STATERES
#   CTELNUM1
#   CELLFON2
#
# contain essentially one value whenever observed.
#
# Example:
#
#       1
#       NaN
#       1
#       NaN
#
# The actual value carries no information.
# The missingness DOES.
#
# Therefore:
#
#       original feature -> feature_missing
#
# ============================================================

INDICATOR_ONLY_FEATURES = {
    "CTELENUM",
    "STATERES",
    "CTELNUM1",
    "CELLFON2",
}

INDICATOR_ONLY_FEATURES &= set(feature_names)


# ============================================================
# 3. ORDINAL FEATURES
# ============================================================
#
# These variables have categories with a meaningful order.
#
# For example GENHLTH:
#
#   1 Excellent
#   2 Very good
#   3 Good
#   4 Fair
#   5 Poor
#
# One-hot encoding would THROW AWAY that known ordering.
#
# We therefore preserve the numerical order.
# ============================================================

ORDINAL_FEATURES = {
    "GENHLTH",
    "CHECKUP1",
    "EDUCA",
    "INCOME2",
    "SMOKDAY2",
    "LASTSMK2",
    "SEATBELT",
}

ORDINAL_FEATURES &= set(feature_names)


# ------------------------------------------------------------
# Codes that sit OUTSIDE the ordinal scale.
#
# CHECKUP1:  1..4 = within 1 / 2 / 5 / 5+ years, 8 = never.
#            Kept as 8, "never" would look 4 steps beyond
#            "5+ years". It is really the next step -> 5.
#
# SEATBELT:  1..5 = always .. never, 8 = never drives.
#            Not a point on this scale at all -> missing.
# ------------------------------------------------------------

ORDINAL_REMAP = {
    "CHECKUP1": {8.0: 5.0},
    "SEATBELT": {8.0: np.nan},
}


def remap_ordinal(col, name):
    """
    Move out-of-scale ordinal codes to their correct position.
    """

    clean = col.copy()

    for old, new in ORDINAL_REMAP.get(name, {}).items():
        clean[col == old] = new

    return clean


# ============================================================
# 4. EXPLICIT NOMINAL CATEGORICAL FEATURES
# ============================================================
#
# Numerical values here are category labels rather than
# quantities.
#
# Example:
#
#   MARITAL = 1,2,3,...
#
# category 4 is NOT "twice" category 2.
#
# These should therefore be dummy encoded.
# ============================================================

NOMINAL_FEATURES = {
    "MARITAL",
    "RENTHOM1",
    "EMPLOY1",

    "EXRACT11",
    "EXRACT21",

    "IMFVPLAC",
    "WHRTST10",

    "CRGVREL1",
    "CRGVPRB1",

    "SXORIENT",
    "TRNSGNDR",

    "MSCODE",

    "_RACE",
    "_RACEGR3",
}

NOMINAL_FEATURES &= set(feature_names)


# ============================================================
# 5. FEATURE-SPECIFIC SPECIAL RESPONSE CODES
# ============================================================
#
# CRITICAL:
#
# We do NOT globally say:
#
#       7 -> missing
#       9 -> missing
#       77 -> missing
#       99 -> missing
#
# because a numerical feature can legitimately contain 7 or 9.
#
# These conversions are applied ONLY to survey-coded
# categorical/ordinal variables.
#
# For most BRFSS single-digit categorical questions:
#
#       7 = Don't know / Not sure
#       9 = Refused
#
# For two-digit responses:
#
#       77 = Don't know
#       99 = Refused
#
# etc.
# ============================================================

# ------------------------------------------------------------
# BUT: 7 and 9 are NOT always missing.
#
# A question with more than 6 real answers needs 7 (and 8, 9)
# for real answers, so the survey moves "don't know/refused"
# to 77/99 instead. Example INCOME2:
#
#       1..8  = income brackets (7 = $50-75k)
#       77/99 = don't know / refused
#
# Rules, checked in this order:
#
#   1. Column in MISSING_CODES_OVERRIDE -> use the listed codes
#   2. Column contains 77 or 99         -> only 77, 99 missing
#   3. Column contains 777 or 999       -> only 777, 999 missing
#   4. Otherwise                        -> 7, 9 missing
#
# The decision is made on TRAINING data only and stored in the
# preprocessing params, so validation/test get the same codes.
# ------------------------------------------------------------

# Single-digit questions where 7 is a REAL answer but 9 still
# means refused. There is no 77/99 in the data to reveal this,
# so they are listed by hand (from the BRFSS codebook).
MISSING_CODES_OVERRIDE = {
    "EMPLOY1": {9.0},   # 7 = retired
    "_RACE":   {9.0},   # 7 = multiracial
}


def find_missing_codes(col, name):
    """
    Decide which codes mean 'don't know / refused' for ONE
    survey-coded categorical/ordinal column.

    Call on TRAINING data only.
    """

    if name in MISSING_CODES_OVERRIDE:
        return MISSING_CODES_OVERRIDE[name]

    observed = set(np.unique(col[~np.isnan(col)]))

    if observed & {77.0, 99.0}:
        return {77.0, 99.0}

    if observed & {777.0, 999.0}:
        return {777.0, 999.0}

    return {7.0, 9.0}


def clean_categorical_codes(col, missing_codes):
    """
    Convert this column's non-response codes to NaN.

    IMPORTANT:
    This function is used only on features we have classified
    as survey-coded categorical/ordinal variables.

    It is NOT applied blindly to continuous numerical columns.
    """

    clean = col.astype(float).copy()

    for code in missing_codes:
        clean[clean == code] = np.nan

    return clean


# ============================================================
# 6. SPECIAL NUMERICAL / COUNT CODINGS
# ============================================================
#
# Some BRFSS variables use codes such as:
#
#       88 = none
#
# rather than "missing".
#
# For variables representing number of days, "none" should
# become zero.
#
# We handle those separately rather than treating 88 as NaN.
# ============================================================

ZERO_88_FEATURES = {
    "PHYSHLTH",
    "MENTHLTH",
    "POORHLTH",
    "AVEDRNK2",
    "DRNK3GE5",
    "MAXDRNKS",
    "CHILDREN",
}

ZERO_88_FEATURES &= set(feature_names)


# Decoded (computed) numerical columns that still use their own
# "don't know / refused" code.
NUMERIC_MISSING_CODES = {
    "DROCDY3_": {900.0},
    "_DRNKWEK": {99900.0},
    "MAXVO2_":  {99900.0},
    "FC60_":    {99900.0},
}


def clean_numerical_feature(col, name):
    """
    Feature-aware cleaning for numerical/count variables.

    We intentionally keep ordinary numbers such as 7 and 9
    because those may be legitimate numerical measurements.

    For selected BRFSS count variables:
        88 -> 0 ('none')

    Very large refusal / unknown codes are converted to NaN
    where appropriate.
    """

    clean = col.astype(float).copy()

    # For these count variables, 88 represents none/zero.
    if name in ZERO_88_FEATURES:
        clean[clean == 88.0] = 0.0

        # Common unknown/refused codes for this type of variable.
        clean[clean == 77.0] = np.nan
        clean[clean == 99.0] = np.nan

    for code in NUMERIC_MISSING_CODES.get(name, set()):
        clean[clean == code] = np.nan

    return clean


# ============================================================
# 7. DROP >=99% MISSING FEATURES
# ============================================================
#
# This decision is based ONLY on train_X.
#
# These variables contain almost no observed training data.
# We drop them in baseline v1.
#
# Later this is an easy ablation:
#
#       threshold = 0.99
#       versus
#       keep them
#
# ============================================================

SPARSE_THRESHOLD = 0.99

extremely_sparse_features = set()

for j, name in enumerate(feature_names):

    # Don't bother analysing something already explicitly
    # dropped.
    if name in DROP_FEATURES:
        continue

    missing_fraction = np.mean(np.isnan(train_X[:, j]))

    if missing_fraction >= SPARSE_THRESHOLD:
        extremely_sparse_features.add(name)


# Indicator-only variables are useful specifically BECAUSE
# missingness is their information, so don't automatically
# remove them here.
extremely_sparse_features -= INDICATOR_ONLY_FEATURES

DROP_FEATURES.update(extremely_sparse_features)


# ============================================================
# 8. AUTOMATIC TYPE CLASSIFICATION FOR REMAINING FEATURES
# ============================================================
#
# We already manually specified:
#
#       ordinal features
#       known nominal features
#       indicator-only features
#
# For the remaining features:
#
#       <= 10 observed unique values
#              -> categorical
#
#       > 10 observed unique values
#              -> numerical
#
# This is appropriate as a baseline for this survey dataset,
# where the majority of low-cardinality columns are encoded
# survey responses.
# ============================================================

MAX_CATEGORIES = 10

CATEGORICAL_FEATURES = set()
NUMERICAL_FEATURES = set()


for j, name in enumerate(feature_names):

    if name in DROP_FEATURES:
        continue

    if name in INDICATOR_ONLY_FEATURES:
        continue

    if name in ORDINAL_FEATURES:
        continue

    if name in NOMINAL_FEATURES:
        CATEGORICAL_FEATURES.add(name)
        continue

    col = train_X[:, j].astype(float)

    observed = col[~np.isnan(col)]

    if len(observed) == 0:
        DROP_FEATURES.add(name)
        continue

    n_unique = len(np.unique(observed))

    if n_unique <= MAX_CATEGORIES:
        CATEGORICAL_FEATURES.add(name)
    else:
        NUMERICAL_FEATURES.add(name)


# ============================================================
# 9. HELPER: FIT NUMERICAL / ORDINAL FEATURE
# ============================================================

def fit_numeric_feature(col):
    """
    Learn imputation/scaling parameters from TRAINING DATA.

    Missing values are replaced with the TRAINING median.

    We additionally preserve a binary missingness indicator,
    so the model can distinguish:

        real median value

    from

        value that was missing and happened to be replaced
        by the median.
    """

    observed = col[~np.isnan(col)]

    if len(observed) == 0:
        # Should normally have been dropped before this point.
        median = 0.0
    else:
        median = np.median(observed)

    imputed = col.copy()
    imputed[np.isnan(imputed)] = median

    mean = np.mean(imputed)
    std = np.std(imputed)

    if std == 0 or np.isnan(std):
        std = 1.0

    return {
        "median": median,
        "mean": mean,
        "std": std,
    }


# ============================================================
# 10. HELPER: TRANSFORM NUMERICAL / ORDINAL FEATURE
# ============================================================

def transform_numeric_feature(col, info):
    """
    Transform one numerical/ordinal column.

    Returns either:

        standardized value

    or:

        standardized value + missing indicator

    We add the indicator only if the TRAINING feature actually
    had missing values.
    """

    missing = np.isnan(col)

    clean = col.copy()
    clean[missing] = info["median"]

    standardized = (
        clean - info["mean"]
    ) / info["std"]

    standardized = standardized.reshape(-1, 1)

    if info["has_missing"]:
        return np.column_stack([
            standardized[:, 0],
            missing.astype(float)
        ])

    return standardized


# ============================================================
# 11. HELPER: CATEGORICAL REPRESENTATION
# ============================================================
#
# Missing gets its own categorical state.
#
# We use a special internal value that cannot clash with a
# genuine BRFSS response.
# ============================================================

MISSING_CATEGORY = -999999.0
UNKNOWN_CATEGORY = -999998.0


def prepare_categorical(col):
    """
    Replace NaN with an explicit internal missing category.
    """

    clean = col.copy()

    clean[np.isnan(clean)] = MISSING_CATEGORY

    return clean


# ============================================================
# 12. FIT CATEGORICAL FEATURE
# ============================================================

def fit_categorical_feature(col):
    """
    Learn categories from TRAINING DATA ONLY.

    We use k-1 dummy encoding rather than k columns.

    Example:

        YES / NO / MISSING

    might become:

                  NO    MISSING
        YES        0       0
        NO         1       0
        MISSING    0       1

    YES is the reference category.

    This avoids exact redundancy when a model has an
    intercept.
    """

    clean = prepare_categorical(col)

    categories = np.unique(clean)

    # Choose the first category as the reference.
    reference = categories[0]

    encoded_categories = categories[1:]

    return {
        "categories": categories,
        "reference": reference,
        "encoded_categories": encoded_categories,
    }


# ============================================================
# 13. TRANSFORM CATEGORICAL FEATURE
# ============================================================

def transform_categorical_feature(col, info):
    """
    Apply training categories to a categorical feature.

    A category that somehow appears in validation/test but
    never appeared in training simply receives zeros across
    all dummy columns, i.e. behaves like the reference level.

    With this dataset, train_X is large enough that this should
    be rare.
    """

    clean = prepare_categorical(col)

    encoded_categories = info["encoded_categories"]

    result = np.zeros(
        (len(clean), len(encoded_categories)),
        dtype=float
    )

    for k, category in enumerate(encoded_categories):
        result[:, k] = (
            clean == category
        ).astype(float)

    return result


# ============================================================
# 14. FIT THE COMPLETE PREPROCESSOR
# ============================================================

def fit_preprocessor(X, feature_names):
    """
    Fit all preprocessing decisions on TRAINING DATA ONLY.

    Returns a dictionary containing everything needed to
    transform validation/test data identically.
    """

    params = {}

    for j, name in enumerate(feature_names):

        # ----------------------------------------------------
        # DROP
        # ----------------------------------------------------

        if name in DROP_FEATURES:
            continue

        col = X[:, j].astype(float).copy()


        # ----------------------------------------------------
        # INDICATOR ONLY
        # ----------------------------------------------------

        if name in INDICATOR_ONLY_FEATURES:

            params[name] = {
                "type": "indicator_only"
            }

            continue


        # ----------------------------------------------------
        # ORDINAL
        # ----------------------------------------------------

        if name in ORDINAL_FEATURES:

            missing_codes = find_missing_codes(col, name)

            clean = clean_categorical_codes(col, missing_codes)

            clean = remap_ordinal(clean, name)

            numeric_info = fit_numeric_feature(clean)

            params[name] = {
                "type": "ordinal",
                "missing_codes": missing_codes,
                **numeric_info,
                "has_missing": np.any(np.isnan(clean)),
            }

            continue


        # ----------------------------------------------------
        # CATEGORICAL
        # ----------------------------------------------------

        if name in CATEGORICAL_FEATURES:

            missing_codes = find_missing_codes(col, name)

            clean = clean_categorical_codes(col, missing_codes)

            categorical_info = fit_categorical_feature(clean)

            params[name] = {
                "type": "categorical",
                "missing_codes": missing_codes,
                **categorical_info,
            }

            continue


        # ----------------------------------------------------
        # NUMERICAL
        # ----------------------------------------------------

        if name in NUMERICAL_FEATURES:

            clean = clean_numerical_feature(
                col,
                name
            )

            numeric_info = fit_numeric_feature(clean)

            params[name] = {
                "type": "numerical",
                **numeric_info,
                "has_missing": np.any(np.isnan(clean)),
            }

            continue


        # ----------------------------------------------------
        # SAFETY FALLBACK
        # ----------------------------------------------------

        raise ValueError(
            f"Feature {name} was not assigned a preprocessing type."
        )

    return params


# ============================================================
# 15. TRANSFORM A COMPLETE DATA MATRIX
# ============================================================

def transform_data(X, feature_names, params):
    """
    Transform a raw BRFSS feature matrix into the final
    numerical design matrix.

    Returns:
        X_processed
        processed_feature_names
    """

    transformed_blocks = []
    output_names = []


    for j, name in enumerate(feature_names):

        # Feature was dropped.
        if name not in params:
            continue

        col = X[:, j].astype(float).copy()

        info = params[name]

        feature_type = info["type"]


        # ====================================================
        # A. INDICATOR ONLY
        # ====================================================

        if feature_type == "indicator_only":

            missing_indicator = np.isnan(
                col
            ).astype(float)

            transformed_blocks.append(
                missing_indicator.reshape(-1, 1)
            )

            output_names.append(
                f"{name}_missing"
            )


        # ====================================================
        # B. ORDINAL
        # ====================================================

        elif feature_type == "ordinal":

            clean = clean_categorical_codes(
                col,
                info["missing_codes"]
            )

            clean = remap_ordinal(clean, name)

            transformed = transform_numeric_feature(
                clean,
                info
            )

            transformed_blocks.append(
                transformed
            )

            output_names.append(name)

            if info["has_missing"]:
                output_names.append(
                    f"{name}_missing"
                )


        # ====================================================
        # C. CATEGORICAL
        # ====================================================

        elif feature_type == "categorical":

            clean = clean_categorical_codes(
                col,
                info["missing_codes"]
            )

            transformed = transform_categorical_feature(
                clean,
                info
            )

            transformed_blocks.append(
                transformed
            )

            for category in info["encoded_categories"]:

                if category == MISSING_CATEGORY:
                    category_label = "MISSING"
                else:
                    # Make integer-looking categories prettier.
                    if float(category).is_integer():
                        category_label = str(
                            int(category)
                        )
                    else:
                        category_label = str(
                            category
                        )

                output_names.append(
                    f"{name}__{category_label}"
                )


        # ====================================================
        # D. NUMERICAL
        # ====================================================

        elif feature_type == "numerical":

            clean = clean_numerical_feature(
                col,
                name
            )

            transformed = transform_numeric_feature(
                clean,
                info
            )

            transformed_blocks.append(
                transformed
            )

            output_names.append(name)

            if info["has_missing"]:
                output_names.append(
                    f"{name}_missing"
                )


        else:
            raise ValueError(
                f"Unknown preprocessing type "
                f"{feature_type} for {name}"
            )


    # Combine every processed feature horizontally.
    X_processed = np.hstack(
        transformed_blocks
    ).astype(float)

    return X_processed, output_names


# ============================================================
# 16. FIT ON train_X ONLY
# ============================================================
#
# THIS is the only dataset from which preprocessing parameters
# are learned.
# ============================================================

preprocessing_params = fit_preprocessor(
    train_X,
    feature_names
)


# ============================================================
# 17. APPLY TO TRAINING DATA
# ============================================================

train_X_processed, processed_feature_names = (
    transform_data(
        train_X,
        feature_names,
        preprocessing_params
    )
)


# ============================================================
# 18. APPLY IDENTICAL TRANSFORMATIONS TO LOCAL VALIDATION
# ============================================================

test_X_processed, validation_feature_names = (
    transform_data(
        test_X,
        feature_names,
        preprocessing_params
    )
)


# ============================================================
# 19. APPLY IDENTICAL TRANSFORMATIONS TO COMPETITION TEST SET
# ============================================================

x_test_processed, competition_feature_names = (
    transform_data(
        x_test,
        feature_names,
        preprocessing_params
    )
)


# ============================================================
# 20. SANITY CHECK: FEATURE ORDER MUST BE IDENTICAL
# ============================================================

assert (
    processed_feature_names
    == validation_feature_names
)

assert (
    processed_feature_names
    == competition_feature_names
)


# ============================================================
# 21. SANITY CHECK: SAME NUMBER OF COLUMNS
# ============================================================

assert (
    train_X_processed.shape[1]
    == test_X_processed.shape[1]
    == x_test_processed.shape[1]
)


# ============================================================
# 22. SANITY CHECK: NO NaNs
# ============================================================

n_nan_train = np.isnan(
    train_X_processed
).sum()

n_nan_validation = np.isnan(
    test_X_processed
).sum()

n_nan_competition = np.isnan(
    x_test_processed
).sum()


assert n_nan_train == 0, (
    f"Training matrix still contains "
    f"{n_nan_train} NaNs."
)

assert n_nan_validation == 0, (
    f"Validation matrix still contains "
    f"{n_nan_validation} NaNs."
)

assert n_nan_competition == 0, (
    f"Competition matrix still contains "
    f"{n_nan_competition} NaNs."
)


# ============================================================
# 23. SANITY CHECK: NO INFINITIES
# ============================================================

assert np.all(
    np.isfinite(train_X_processed)
)

assert np.all(
    np.isfinite(test_X_processed)
)

assert np.all(
    np.isfinite(x_test_processed)
)


# ============================================================
# 24. PRINT PREPROCESSING SUMMARY
# ============================================================

print()
print("=" * 70)
print("PREPROCESSING COMPLETE")
print("=" * 70)

print("\nORIGINAL DATA")
print("-" * 70)

print(
    f"Training:          {train_X.shape}"
)

print(
    f"Local validation:  {test_X.shape}"
)

print(
    f"Competition test:  {x_test.shape}"
)


print("\nPROCESSED DATA")
print("-" * 70)

print(
    f"Training:          {train_X_processed.shape}"
)

print(
    f"Local validation:  {test_X_processed.shape}"
)

print(
    f"Competition test:  {x_test_processed.shape}"
)


print("\nFEATURE COUNTS")
print("-" * 70)

print(
    f"Original features:       "
    f"{train_X.shape[1]}"
)

print(
    f"Final processed columns: "
    f"{train_X_processed.shape[1]}"
)

print(
    f"Dropped features:        "
    f"{len(DROP_FEATURES)}"
)

print(
    f"Indicator-only features: "
    f"{len(INDICATOR_ONLY_FEATURES)}"
)

print(
    f"Ordinal features:        "
    f"{len(ORDINAL_FEATURES)}"
)

print(
    f"Categorical features:    "
    f"{len(CATEGORICAL_FEATURES)}"
)

print(
    f"Numerical features:      "
    f"{len(NUMERICAL_FEATURES)}"
)

print(
    f">=99% missing dropped:   "
    f"{len(extremely_sparse_features)}"
)


print("\nMISSING VALUES AFTER PREPROCESSING")
print("-" * 70)

print(
    f"Training:          {n_nan_train}"
)

print(
    f"Local validation:  {n_nan_validation}"
)

print(
    f"Competition test:  {n_nan_competition}"
)


# ============================================================
# 25. SHOW EXACTLY WHAT WAS DROPPED
# ============================================================

print("\nDROPPED FEATURES")
print("-" * 70)

for name in sorted(DROP_FEATURES):
    print(name)


# ============================================================
# 25b. CHECK NUMERICAL COLUMNS FOR LEFTOVER SPECIAL CODES
# ============================================================
#
# If a numerical column's maximum is a typical BRFSS code
# (777, 999, 99900, ...), it probably still contains
# "don't know / refused" answers. Look it up in the codebook
# and add it to NUMERIC_MISSING_CODES.
# ============================================================

SUSPICIOUS_MAXIMA = {
    77.0, 88.0, 99.0,
    777.0, 888.0, 900.0, 999.0,
    7777.0, 9999.0,
    99900.0,
}

print("\nNUMERICAL COLUMNS WITH SUSPICIOUS MAXIMUM")
print("-" * 70)

n_suspicious = 0

for name, info in preprocessing_params.items():

    if info["type"] != "numerical":
        continue

    col = clean_numerical_feature(
        train_X[:, name_to_index[name]],
        name
    )

    col_max = np.nanmax(col)

    if col_max in SUSPICIOUS_MAXIMA:
        n_suspicious += 1
        print(f"{name:12s} max = {col_max:g}  <- check codebook")

if n_suspicious == 0:
    print("None found.")


# ============================================================
# 26. SHOW FINAL FEATURE NAMES
# ============================================================
#
# This is important because after dummy encoding:
#
#       321 original columns
#
# will become some different number of processed columns.
#
# We preserve the exact meaning/order of every resulting
# column.
# ============================================================

print("\nFINAL PROCESSED FEATURES")
print("-" * 70)

for i, name in enumerate(
    processed_feature_names
):
    print(
        f"{i:4d}: {name}"
    )


# ============================================================
# 27. OPTIONAL: CONVERT LABELS FOR LOGISTIC REGRESSION
# ============================================================
#
# Your dataset labels are:
#
#       -1 = negative
#       +1 = positive
#
# Your logistic-regression implementation expects:
#
#        0 = negative
#        1 = positive
#
# Keep train_y/test_y unchanged for other uses and create
# separate logistic-regression versions.
# ============================================================

train_y_logistic = (
    train_y == 1
).astype(int)

test_y_logistic = (
    test_y == 1
).astype(int)


print("\nLABEL CHECK")
print("-" * 70)

print(
    "Original training labels:",
    np.unique(train_y)
)

print(
    "Logistic training labels:",
    np.unique(train_y_logistic)
)


# ============================================================
# DONE
# ============================================================
#
# From this point onward:
#
#   TRAINING FEATURES:
#       train_X_processed
#
#   TRAINING LABELS:
#       train_y
#       or train_y_logistic for logistic regression
#
#   LOCAL VALIDATION FEATURES:
#       test_X_processed
#
#   LOCAL VALIDATION LABELS:
#       test_y
#       or test_y_logistic for logistic regression
#
#   COMPETITION TEST FEATURES:
#       x_test_processed
#
#   FINAL FEATURE NAMES:
#       processed_feature_names
#
# ============================================================

Raw number of features: 321



PREPROCESSING COMPLETE

ORIGINAL DATA
----------------------------------------------------------------------
Training:          (262508, 321)
Local validation:  (65627, 321)
Competition test:  (109379, 321)

PROCESSED DATA
----------------------------------------------------------------------
Training:          (262508, 836)
Local validation:  (65627, 836)
Competition test:  (109379, 836)

FEATURE COUNTS
----------------------------------------------------------------------
Original features:       321
Final processed columns: 836
Dropped features:        70
Indicator-only features: 4
Ordinal features:        7
Categorical features:    180
Numerical features:      60
>=99% missing dropped:   27

MISSING VALUES AFTER PREPROCESSING
----------------------------------------------------------------------
Training:          0
Local validation:  0
Competition test:  0

DROPPED FEATURES
----------------------------------------------------------------------
ALCDAY5
ASACTLIM
ASATTACK
ASDRVIST
A

### Training

In [15]:
# ============================================================
# REGULARIZED LOGISTIC REGRESSION
# HYPERPARAMETER + THRESHOLD SEARCH
# ============================================================
#
# For every combination of:
#
#       lambda
#       gamma
#       max_iters
#
# we:
#
#   1. Train logistic regression on TRAINING data.
#   2. Compute probabilities on TRAINING and VALIDATION.
#   3. Search many classification thresholds.
#   4. Find the threshold giving the best VALIDATION F1.
#   5. Store all results.
#
# Finally:
#
#   - rank all models by validation F1
#   - print the best hyperparameters
#   - print train/validation performance of the winner
#
# IMPORTANT:
#
# Competition x_test is NOT touched here.
# ============================================================


import numpy as np
import time

from implementations import (
    reg_logistic_regression,
    sigmoid
)


# ============================================================
# 1. PREPARE MATRICES
# ============================================================

tx_train = np.column_stack([
    np.ones(train_X_processed.shape[0]),
    train_X_processed
])

tx_validation = np.column_stack([
    np.ones(test_X_processed.shape[0]),
    test_X_processed
])


y_tr = train_y_logistic
y_val = test_y_logistic


print("Training matrix:", tx_train.shape)
print("Validation matrix:", tx_validation.shape)

print(
    f"Training positive fraction: "
    f"{np.mean(y_tr == 1):.4f}"
)

print(
    f"Validation positive fraction: "
    f"{np.mean(y_val == 1):.4f}"
)


# ============================================================
# 2. METRIC FUNCTION
# ============================================================

def get_classification_metrics(
    y_true,
    probabilities,
    threshold
):

    y_pred = (
        probabilities >= threshold
    ).astype(int)


    tp = np.sum(
        (y_true == 1) &
        (y_pred == 1)
    )

    tn = np.sum(
        (y_true == 0) &
        (y_pred == 0)
    )

    fp = np.sum(
        (y_true == 0) &
        (y_pred == 1)
    )

    fn = np.sum(
        (y_true == 1) &
        (y_pred == 0)
    )


    accuracy = (
        (tp + tn)
        /
        len(y_true)
    )


    if (tp + fp) > 0:
        precision = tp / (tp + fp)
    else:
        precision = 0.0


    if (tp + fn) > 0:
        recall = tp / (tp + fn)
    else:
        recall = 0.0


    if (precision + recall) > 0:
        f1 = (
            2
            * precision
            * recall
            /
            (precision + recall)
        )
    else:
        f1 = 0.0


    predicted_positive_fraction = np.mean(
        y_pred == 1
    )


    return {
        "threshold": threshold,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "predicted_positive_fraction":
            predicted_positive_fraction,
    }


# ============================================================
# 3. THRESHOLD SEARCH FUNCTION
# ============================================================
#
# Our previous experiment showed the optimum around 0.19.
#
# This time we search more precisely:
#
#       0.05 -> 0.40
#
# in increments of:
#
#       0.005
#
# So we can find thresholds such as:
#
#       0.185
#       0.190
#       0.195
#
# rather than only multiples of 0.01.
# ============================================================

thresholds = np.arange(
    0.05,
    0.4001,
    0.005
)


def find_best_threshold(
    y_true,
    probabilities
):

    best_result = None


    for threshold in thresholds:

        result = get_classification_metrics(
            y_true,
            probabilities,
            threshold
        )


        if (
            best_result is None
            or
            result["f1"] > best_result["f1"]
        ):

            best_result = result


    return best_result


# ============================================================
# 4. HYPERPARAMETER GRID
# ============================================================
#
# History:
#
#   1. Grid 250/500/1000 iterations, gamma up to 0.3:
#      best models always used the LARGEST gamma and the
#      MOST iterations -> training had not finished.
#
#   2. Test run with 3000 iterations:
#      lambda = 1e-4, gamma = 0.6 -> validation F1 = 0.4430
#
# This grid keeps all previous values and adds the new ones
# (gamma 0.6, 3000 iterations).
#
# Total models:
#
#       3 lambdas
#     x 4 gammas
#     x 3 iteration counts
#     ---------------------
#       36 models
#
# WARNING:
#
# This takes roughly 2-2.5 hours: every iteration performs a
# matrix multiplication over ~262k x 836 values, and the
# 3000-iteration models alone take ~7-8 minutes each.
# ============================================================

lambda_values = [
    1e-5,
    1e-4,
    1e-3,
]


gamma_values = [
    0.03,
    0.1,
    0.3,
    0.6,
]


iteration_values = [
    500,
    1000,
    3000,
]


total_models = (
    len(lambda_values)
    *
    len(gamma_values)
    *
    len(iteration_values)
)


print("\n")
print("=" * 90)
print("HYPERPARAMETER SEARCH")
print("=" * 90)

print(
    f"\nNumber of models to train: "
    f"{total_models}"
)

print(
    f"Number of thresholds per model: "
    f"{len(thresholds)}"
)


# ============================================================
# 5. RUN GRID SEARCH
# ============================================================

all_results = []

model_number = 0

overall_start_time = time.time()


for lambda_ in lambda_values:

    for gamma in gamma_values:

        for max_iters in iteration_values:

            model_number += 1


            print("\n")
            print("=" * 90)

            print(
                f"MODEL "
                f"{model_number}/{total_models}"
            )

            print("=" * 90)


            print(
                f"lambda     = {lambda_}"
            )

            print(
                f"gamma      = {gamma}"
            )

            print(
                f"max_iters  = {max_iters}"
            )


            # ==================================================
            # INITIALIZE WEIGHTS
            # ==================================================

            initial_w = np.zeros(
                tx_train.shape[1]
            )


            # ==================================================
            # TRAIN MODEL
            # ==================================================

            start_time = time.time()


            try:

                w, loss = reg_logistic_regression(
                    y_tr,
                    tx_train,
                    lambda_,
                    initial_w,
                    max_iters,
                    gamma
                )


            except Exception as e:

                print(
                    "\nMODEL FAILED:"
                )

                print(e)

                continue


            training_time = (
                time.time()
                -
                start_time
            )


            # ==================================================
            # CHECK FOR NUMERICAL FAILURE
            # ==================================================

            if (
                np.any(np.isnan(w))
                or
                np.any(np.isinf(w))
                or
                np.isnan(loss)
                or
                np.isinf(loss)
            ):

                print(
                    "\nNumerical instability detected."
                )

                print(
                    "Skipping this configuration."
                )

                continue


            # ==================================================
            # CALCULATE PROBABILITIES
            # ==================================================

            train_prob = sigmoid(
                tx_train @ w
            )

            validation_prob = sigmoid(
                tx_validation @ w
            )


            # ==================================================
            # CHECK PROBABILITIES
            # ==================================================

            if (
                np.any(np.isnan(train_prob))
                or
                np.any(np.isnan(validation_prob))
            ):

                print(
                    "\nNaN probabilities detected."
                )

                print(
                    "Skipping this configuration."
                )

                continue


            # ==================================================
            # FIND BEST VALIDATION THRESHOLD
            # ==================================================

            best_validation = (
                find_best_threshold(
                    y_val,
                    validation_prob
                )
            )


            best_threshold = (
                best_validation["threshold"]
            )


            # ==================================================
            # TRAIN PERFORMANCE AT SAME THRESHOLD
            # ==================================================

            train_result = (
                get_classification_metrics(
                    y_tr,
                    train_prob,
                    best_threshold
                )
            )


            # ==================================================
            # SAVE RESULT
            # ==================================================

            result = {

                "lambda":
                    lambda_,

                "gamma":
                    gamma,

                "max_iters":
                    max_iters,

                "loss":
                    loss,

                "threshold":
                    best_threshold,

                "train_accuracy":
                    train_result["accuracy"],

                "train_precision":
                    train_result["precision"],

                "train_recall":
                    train_result["recall"],

                "train_f1":
                    train_result["f1"],

                "validation_accuracy":
                    best_validation["accuracy"],

                "validation_precision":
                    best_validation["precision"],

                "validation_recall":
                    best_validation["recall"],

                "validation_f1":
                    best_validation["f1"],

                "tp":
                    best_validation["tp"],

                "tn":
                    best_validation["tn"],

                "fp":
                    best_validation["fp"],

                "fn":
                    best_validation["fn"],

                "predicted_positive_fraction":
                    best_validation[
                        "predicted_positive_fraction"
                    ],

                "training_time":
                    training_time,

                "weights":
                    w.copy(),
            }


            all_results.append(
                result
            )


            # ==================================================
            # PRINT CURRENT RESULT
            # ==================================================

            print(
                f"\nTraining loss: "
                f"{loss:.6f}"
            )

            print(
                f"Training time: "
                f"{training_time:.1f} seconds"
            )

            print(
                f"\nBest threshold: "
                f"{best_threshold:.3f}"
            )


            print(
                "\nVALIDATION"
            )

            print(
                f"Accuracy:  "
                f"{best_validation['accuracy']:.4f}"
            )

            print(
                f"Precision: "
                f"{best_validation['precision']:.4f}"
            )

            print(
                f"Recall:    "
                f"{best_validation['recall']:.4f}"
            )

            print(
                f"F1:        "
                f"{best_validation['f1']:.4f}"
            )


            print(
                "\nTRAINING"
            )

            print(
                f"Accuracy:  "
                f"{train_result['accuracy']:.4f}"
            )

            print(
                f"Precision: "
                f"{train_result['precision']:.4f}"
            )

            print(
                f"Recall:    "
                f"{train_result['recall']:.4f}"
            )

            print(
                f"F1:        "
                f"{train_result['f1']:.4f}"
            )


            print(
                f"\nF1 gap "
                f"(train - validation): "
                f"{train_result['f1'] - best_validation['f1']:.4f}"
            )


            # ==================================================
            # CURRENT BEST MODEL
            # ==================================================

            current_best = max(
                all_results,
                key=lambda r:
                    r["validation_f1"]
            )


            print(
                "\nCURRENT BEST"
            )

            print(
                f"F1 = "
                f"{current_best['validation_f1']:.4f}"
            )

            print(
                f"lambda = "
                f"{current_best['lambda']}"
            )

            print(
                f"gamma = "
                f"{current_best['gamma']}"
            )

            print(
                f"iterations = "
                f"{current_best['max_iters']}"
            )

            print(
                f"threshold = "
                f"{current_best['threshold']:.3f}"
            )


# ============================================================
# 6. TOTAL SEARCH TIME
# ============================================================

overall_time = (
    time.time()
    -
    overall_start_time
)


print("\n")
print("=" * 100)
print("GRID SEARCH COMPLETE")
print("=" * 100)


print(
    f"\nTotal search time: "
    f"{overall_time / 60:.2f} minutes"
)


print(
    f"Successful models: "
    f"{len(all_results)} / {total_models}"
)


# ============================================================
# 7. SORT MODELS BY VALIDATION F1
# ============================================================

all_results_sorted = sorted(
    all_results,
    key=lambda r:
        r["validation_f1"],
    reverse=True
)


# ============================================================
# 8. PRINT RANKING
# ============================================================

print("\n")
print("=" * 130)
print("MODEL RANKING — SORTED BY VALIDATION F1")
print("=" * 130)


print(
    f"{'Rank':>5}"
    f"{'Lambda':>12}"
    f"{'Gamma':>10}"
    f"{'Iters':>10}"
    f"{'Thresh':>10}"
    f"{'Train F1':>12}"
    f"{'Val F1':>12}"
    f"{'Val Acc':>12}"
    f"{'Precision':>12}"
    f"{'Recall':>12}"
)


print("-" * 130)


for rank, result in enumerate(
    all_results_sorted,
    start=1
):

    print(
        f"{rank:5d}"
        f"{result['lambda']:12.1e}"
        f"{result['gamma']:10.3f}"
        f"{result['max_iters']:10d}"
        f"{result['threshold']:10.3f}"
        f"{result['train_f1']:12.4f}"
        f"{result['validation_f1']:12.4f}"
        f"{result['validation_accuracy']:12.4f}"
        f"{result['validation_precision']:12.4f}"
        f"{result['validation_recall']:12.4f}"
    )


# ============================================================
# 9. GET WINNING MODEL
# ============================================================

best_model = (
    all_results_sorted[0]
)


best_lambda = (
    best_model["lambda"]
)

best_gamma = (
    best_model["gamma"]
)

best_max_iters = (
    best_model["max_iters"]
)

best_threshold = (
    best_model["threshold"]
)

best_weights = (
    best_model["weights"]
)


# ============================================================
# 10. PRINT WINNER
# ============================================================

print("\n")
print("=" * 90)
print("BEST MODEL")
print("=" * 90)


print(
    f"\nLambda:     "
    f"{best_lambda}"
)

print(
    f"Gamma:      "
    f"{best_gamma}"
)

print(
    f"Iterations: "
    f"{best_max_iters}"
)

print(
    f"Threshold:  "
    f"{best_threshold:.3f}"
)

print(
    f"Loss:       "
    f"{best_model['loss']:.6f}"
)


print("\nTRAINING")
print("-" * 90)


print(
    f"Accuracy:  "
    f"{best_model['train_accuracy']:.4f}"
)

print(
    f"Precision: "
    f"{best_model['train_precision']:.4f}"
)

print(
    f"Recall:    "
    f"{best_model['train_recall']:.4f}"
)

print(
    f"F1:        "
    f"{best_model['train_f1']:.4f}"
)


print("\nVALIDATION")
print("-" * 90)


print(
    f"Accuracy:  "
    f"{best_model['validation_accuracy']:.4f}"
)

print(
    f"Precision: "
    f"{best_model['validation_precision']:.4f}"
)

print(
    f"Recall:    "
    f"{best_model['validation_recall']:.4f}"
)

print(
    f"F1:        "
    f"{best_model['validation_f1']:.4f}"
)


print(
    f"\nTP={best_model['tp']}, "
    f"TN={best_model['tn']}, "
    f"FP={best_model['fp']}, "
    f"FN={best_model['fn']}"
)


print(
    f"\nPredicted positive fraction: "
    f"{100 * best_model['predicted_positive_fraction']:.2f}%"
)


print(
    f"\nTrain-validation F1 gap: "
    f"{best_model['train_f1'] - best_model['validation_f1']:.4f}"
)


# ============================================================
# 11. COMPARE AGAINST ORIGINAL BASELINE
# ============================================================
#
# Previous best (cleaned preprocessing, 3000 iterations,
# lambda = 1e-4, gamma = 0.6):
#
#       Validation F1 = 0.4430
#
# at threshold:
#
#       0.315
# ============================================================

original_f1 = 0.4430


improvement = (
    best_model["validation_f1"]
    -
    original_f1
)


print("\n")
print("=" * 90)
print("COMPARISON WITH ORIGINAL MODEL")
print("=" * 90)


print(
    f"\nOriginal validation F1: "
    f"{original_f1:.4f}"
)

print(
    f"Best validation F1:     "
    f"{best_model['validation_f1']:.4f}"
)

print(
    f"Absolute improvement:   "
    f"{improvement:+.4f}"
)


# ============================================================
# 12. SAVE FINAL VALUES
# ============================================================
#
# These variables can later be used when we retrain on ALL
# labeled data and predict the competition test set.
# ============================================================

optimized_lambda = best_lambda

optimized_gamma = best_gamma

optimized_max_iters = best_max_iters

optimized_threshold = best_threshold

optimized_weights = best_weights


print("\n")
print("=" * 90)
print("FINAL SELECTED CONFIGURATION")
print("=" * 90)


print(
    f"\nlambda     = "
    f"{optimized_lambda}"
)

print(
    f"gamma      = "
    f"{optimized_gamma}"
)

print(
    f"max_iters  = "
    f"{optimized_max_iters}"
)

print(
    f"threshold  = "
    f"{optimized_threshold:.3f}"
)

print(
    f"\nValidation F1 = "
    f"{best_model['validation_f1']:.4f}"
)

Training matrix: (262508, 837)
Validation matrix: (65627, 837)
Training positive fraction: 0.0877
Validation positive fraction: 0.0907


HYPERPARAMETER SEARCH

Number of models to train: 36
Number of thresholds per model: 71


MODEL 1/36
lambda     = 1e-05
gamma      = 0.03
max_iters  = 500



Training loss: 0.227850
Training time: 72.5 seconds

Best threshold: 0.180

VALIDATION
Accuracy:  0.8596
Precision: 0.3339
Recall:    0.5507
F1:        0.4157

TRAINING
Accuracy:  0.8589
Precision: 0.3216
Recall:    0.5485
F1:        0.4054

F1 gap (train - validation): -0.0103

CURRENT BEST
F1 = 0.4157
lambda = 1e-05
gamma = 0.03
iterations = 500
threshold = 0.180


MODEL 2/36
lambda     = 1e-05
gamma      = 0.03
max_iters  = 1000



Training loss: 0.224554
Training time: 143.5 seconds

Best threshold: 0.180

VALIDATION
Accuracy:  0.8607
Precision: 0.3408
Recall:    0.5727
F1:        0.4273

TRAINING
Accuracy:  0.8590
Precision: 0.3256
Recall:    0.5677
F1:        0.4139

F1 gap (train - validation): -0.0135

CURRENT BEST
F1 = 0.4273
lambda = 1e-05
gamma = 0.03
iterations = 1000
threshold = 0.180


MODEL 3/36
lambda     = 1e-05
gamma      = 0.03
max_iters  = 3000



Training loss: 0.221136
Training time: 430.0 seconds

Best threshold: 0.190

VALIDATION
Accuracy:  0.8686
Precision: 0.3576
Recall:    0.5630
F1:        0.4374

TRAINING
Accuracy:  0.8672
Precision: 0.3420
Recall:    0.5561
F1:        0.4235

F1 gap (train - validation): -0.0139

CURRENT BEST
F1 = 0.4374
lambda = 1e-05
gamma = 0.03
iterations = 3000
threshold = 0.190


MODEL 4/36
lambda     = 1e-05
gamma      = 0.1
max_iters  = 500



Training loss: 0.222762
Training time: 71.9 seconds

Best threshold: 0.195

VALIDATION
Accuracy:  0.8702
Precision: 0.3583
Recall:    0.5443
F1:        0.4322

TRAINING
Accuracy:  0.8685
Precision: 0.3418
Recall:    0.5396
F1:        0.4185

F1 gap (train - validation): -0.0137

CURRENT BEST
F1 = 0.4374
lambda = 1e-05
gamma = 0.03
iterations = 3000
threshold = 0.190


MODEL 5/36
lambda     = 1e-05
gamma      = 0.1
max_iters  = 1000


...


Training loss: 0.219890
Training time: 143.2 seconds

Best threshold: 0.190

VALIDATION
Accuracy:  0.8692
Precision: 0.3593
Recall:    0.5640
F1:        0.4390

TRAINING
Accuracy:  0.8688
Precision: 0.3467
Recall:    0.5606
F1:        0.4284

F1 gap (train - validation): -0.0106

CURRENT BEST
F1 = 0.4430
lambda = 0.0001
gamma = 0.6
iterations = 3000
threshold = 0.315


MODEL 33/36
lambda     = 0.001
gamma      = 0.3
max_iters  = 3000



Training loss: 0.219530
Training time: 428.7 seconds

Best threshold: 0.195

VALIDATION
Accuracy:  0.8725
Precision: 0.3663
Recall:    0.5553
F1:        0.4414

TRAINING
Accuracy:  0.8716
Precision: 0.3520
Recall:    0.5515
F1:        0.4297

F1 gap (train - validation): -0.0117

CURRENT BEST
F1 = 0.4430
lambda = 0.0001
gamma = 0.6
iterations = 3000
threshold = 0.315


MODEL 34/36
lambda     = 0.001
gamma      = 0.6
max_iters  = 500



Training loss: 0.346977
Training time: 71.6 seconds

Best threshold: 0.050

VALIDATION
Accuracy:  0.9089
Precision: 0.4959
Recall:    0.2718
F1:        0.3511

TRAINING
Accuracy:  0.9103
Precision: 0.4791
Recall:    0.2674
F1:        0.3432

F1 gap (train - validation): -0.0079

CURRENT BEST
F1 = 0.4430
lambda = 0.0001
gamma = 0.6
iterations = 3000
threshold = 0.315


MODEL 35/36
lambda     = 0.001
gamma      = 0.6
max_iters  = 1000



Training loss: 0.238867
Training time: 143.1 seconds

Best threshold: 0.090

VALIDATION
Accuracy:  0.8677
Precision: 0.3572
Recall:    0.5724
F1:        0.4399

TRAINING
Accuracy:  0.8669
Precision: 0.3430
Recall:    0.5651
F1:        0.4269

F1 gap (train - validation): -0.0130

CURRENT BEST
F1 = 0.4430
lambda = 0.0001
gamma = 0.6
iterations = 3000
threshold = 0.315


MODEL 36/36
lambda     = 0.001
gamma      = 0.6
max_iters  = 3000



Training loss: 0.389900
Training time: 428.4 seconds

Best threshold: 0.050

VALIDATION
Accuracy:  0.9131
Precision: 0.5701
Recall:    0.1701
F1:        0.2621

TRAINING
Accuracy:  0.9148
Precision: 0.5467
Recall:    0.1673
F1:        0.2562

F1 gap (train - validation): -0.0058

CURRENT BEST
F1 = 0.4430
lambda = 0.0001
gamma = 0.6
iterations = 3000
threshold = 0.315


GRID SEARCH COMPLETE

Total search time: 129.10 minutes
Successful models: 36 / 36


MODEL RANKING — SORTED BY VALIDATION F1
 Rank      Lambda     Gamma     Iters    Thresh    Train F1      Val F1     Val Acc   Precision      Recall
----------------------------------------------------------------------------------------------------------------------------------
    1     1.0e-04     0.600      3000     0.315      0.4324      0.4430      0.8733      0.3684      0.5556
    2     1.0e-04     0.300      3000     0.205      0.4320      0.4427      0.8767      0.3752      0.5400
    3     1.0e-05     0.300      3000     0.200